# Warehouses Data Cleaning

## Data Preprocessing Workflow

This notebook follows the same workflow described in the preprocessing PDF: inspection, cleaning, validation, and final transformation.

Dataset: `warehouses.csv`

Warehouse master data must map to valid locations and have positive operational capacity values.

The goal is to move raw data into a clean and reliable format before analysis, modeling, or database ingestion.


## 1. Data Inspection

Before cleaning, inspect the dataset to understand its size, structure, missing values, duplicates, and general quality.


In [ ]:
from pathlib import Path
import pandas as pd

candidate = Path.cwd().resolve()
root = candidate
for parent in [candidate, *candidate.parents]:
    if (parent / 'README.md').exists() and (parent / 'Datasets' / 'raw').exists():
        root = parent
        break
file_path = root / 'Datasets' / 'processed' / 'warehouses.csv'
df = pd.read_csv(file_path)

print(f'Loaded: {file_path.name}')
print(f'Shape: {df.shape}')
print(f'Columns and dtypes:\n{df.dtypes}')
print(f'Head:\n{df.head().to_string(index=False)}')
print(f'Missing values:\n{df.isnull().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique counts:\n{df.nunique()}')


## 2. Data Cleaning Checklist

The following 12 factors must be checked one by one before final preprocessing.


## 1. Duplicate Records

- What to check: Same ID appears twice
- Action: Remove duplicates


In [ ]:
dup_mask = df.duplicated(subset=['warehouse_id'], keep=False)
dup_rows = df.loc[dup_mask]
print(f'Duplicate primary key rows: {len(dup_rows)}')
dup_rows.head() if not dup_rows.empty else print('No duplicate rows found.')


## 2. Missing Values

- What to check: Blank or NULL fields
- Action: Fill or reject


In [ ]:
missing = df.isna().sum().to_frame(name='missing_values')
missing = missing[missing['missing_values'] > 0]
print(missing) if not missing.empty else print('No missing values found.')


## 3. Primary ID Uniqueness

- What to check: product_id, warehouse_id etc.
- Action: Must be unique


In [ ]:
unique_count = df['warehouse_id'].nunique()
print(f'Unique warehouse_id values: {unique_count}')
print(f'Total rows: {len(df)}')
bad = df[df['warehouse_id'].duplicated(keep=False)]
bad.head() if not bad.empty else print('Primary key is unique across the dataset.')


## 4. Reference Integrity

- What to check: Invalid product_id in Sales
- Action: Flag error


In [ ]:
valid_locations = pd.read_csv(root / 'Datasets' / 'processed' / 'locations.csv')['location_id']
bad_locations = df[~df['location_id'].isin(valid_locations)]
print(f'Invalid location references: {len(bad_locations)}')
bad_locations.head() if not bad_locations.empty else print('No invalid location references.')


## 5. Date Format

- What to check: Mixed date formats
- Action: Convert to YYYY-MM-DD


In [ ]:
print('This dataset does not contain date columns that require formatting cleanup.')


## 6. Data Type

- What to check: Text in numeric columns
- Action: Convert to correct type


In [ ]:
for col in ['capacity_units', 'daily_dispatch_capacity_units']:
    converted = pd.to_numeric(df[col], errors='coerce')
    bad = df[col].notna() & converted.isna()
    print(f'{col}: non-numeric values = {bad.sum()}')


## 7. Whitespace & Text

- What to check: Extra spaces, inconsistent names
- Action: Trim & standardize


In [ ]:
for col in ['warehouse_name', 'area', 'city', 'location_id']:
    trimmed = df[col].astype(str).str.strip()
    changed = (trimmed != df[col].astype(str)).sum()
    print(f'{col}: whitespace changes = {changed}')


## 8. Coordinate Validation

- What to check: Latitude/Longitude range
- Action: Validate values


In [ ]:
print('Coordinate validation is not applicable to this dataset because no latitude/longitude fields are present.')


## 9. Numeric Range

- What to check: Negative stock or price
- Action: Correct or reject


In [ ]:
for col in ['capacity_units', 'daily_dispatch_capacity_units']:
    negative = df[col][df[col] < 0]
    print(f'{col}: negative values = {len(negative)}')


## 10. Shelf Validation

- What to check: Empty or duplicate shelf_id
- Action: Make mandatory


In [ ]:
print('Shelf validation is not applicable for this dataset.')


## 11. Category Consistency

- What to check: Different spellings of categories
- Action: Standardize


In [ ]:
for col in ['area', 'city']:
    values = df[col].dropna().astype(str).str.strip().unique()[:10]
    print(f'{col} sample values: {list(values)}')


## 12. Week Consistency

- What to check: Missing weekly records
- Action: Ensure continuous timeline


In [ ]:
print('Week consistency is not applicable for this dataset.')


## 13. Data Validation

After cleaning, validate that the data still follows the expected rules and relationships.


In [ ]:
cleaned = df.copy()
cleaned = cleaned.drop_duplicates()
cleaned = cleaned.dropna(subset=[next(iter(cleaned.columns))]) if len(cleaned.columns) > 0 else cleaned
print(f'After duplicate removal: {cleaned.shape}')
print(cleaned.head().to_string(index=False))


## 14. Data Transformation

This step prepares the cleaned data for modeling or downstream database use. The transformation may include type conversion, date normalization, trimming, and schema standardization.


In [ ]:
clean_df = df.copy()
clean_df = clean_df.drop_duplicates()
for col in clean_df.columns:
    if clean_df[col].dtype == 'object':
        clean_df[col] = clean_df[col].astype(str).str.strip()
        clean_df[col] = clean_df[col].replace({'nan': None, 'None': None})

for col in ['cost_price', 'selling_price', 'capacity_units', 'daily_dispatch_capacity_units', 'units_sold', 'avg_selling_price_rs', 'revenue_rs', 'temperature_mean_c', 'rainfall_mm', 'humidity_pct', 'year']:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

if 'event_date' in clean_df.columns:
    clean_df['event_date'] = pd.to_datetime(clean_df['event_date'], errors='coerce').dt.strftime('%Y-%m-%d')
print('Final cleaned preview:')
print(clean_df.head().to_string(index=False))


## Final Decision

If every quality check passes, keep the cleaned rows and finalize the processed dataset for downstream use.

If a check fails, either correct the values or reject the affected rows before writing the final cleaned CSV.
